In [3]:
# --- Install deps (run once per Colab session) ---
!pip install geopandas requests

# --- Mount Google Drive (for AOI + building files) ---
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import io
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ------------- CONFIG -------------
FIRMS_KEY = ""

# Paths in your Google Drive (EDIT THESE)
AOI_FILE = "/content/drive/MyDrive/Wildfires_Demo/Wildfire_AOI.geojson"
BUILDINGS_FILE = "/content/drive/MyDrive/Wildfires_Demo/Wildfire_Demo_Infrastructure.geojson"

# Use NOAA-21, which matches the noaa21_24hrs layer in QGIS
FIRMS_SOURCE = "VIIRS_NOAA21_NRT"   # NOAA-21 NRT/URT
DAY_RANGE = 2                       # last 2 days (safer than strict 'today')
BUFFER_KM = 5.689                       # fire buffer radius

# ------------- LOAD AOI + BUILDINGS -------------
aoi = gpd.read_file(AOI_FILE)
if len(aoi) > 1:
    aoi = aoi.dissolve()    # single polygon if multiple features

# AOI to WGS84 for bbox + clipping
aoi = aoi.to_crs("EPSG:4326")
minx, miny, maxx, maxy = aoi.total_bounds
bbox_str = f"{minx},{miny},{maxx},{maxy}"

buildings = gpd.read_file(BUILDINGS_FILE)
buildings = buildings.to_crs("EPSG:3857")   # metric CRS for distance

print("AOI bounds (lon/lat):", bbox_str)
print("Number of buildings:", len(buildings))

# ------------- FETCH FIRMS DATA -------------
url = (
    f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
    f"{FIRMS_KEY}/{FIRMS_SOURCE}/{bbox_str}/{DAY_RANGE}"
)
print("Requesting FIRMS data from:", url)

r = requests.get(url)
r.raise_for_status()

df = pd.read_csv(io.StringIO(r.text))

if df.empty:
    print("No FIRMS detections from NOAA-21 in AOI bbox over last", DAY_RANGE, "day(s).")
else:
    # Make GeoDataFrame from FIRMS points
    gdf_fires = gpd.GeoDataFrame(
        df,
        geometry=[Point(xy) for xy in zip(df["longitude"], df["latitude"])],
        crs="EPSG:4326",
    )

    # Clip to AOI polygon (extra safety)
    gdf_fires = gpd.clip(gdf_fires, aoi)

    if gdf_fires.empty:
        print("FIRMS returned points, but none fall inside AOI polygon.")
    else:
        print("Number of FIRMS detections in AOI:", len(gdf_fires))

        # Reproject to metric CRS for buffering
        fires_3857 = gdf_fires.to_crs("EPSG:3857")

        # Buffer fires by 5 km
        buffer_m = BUFFER_KM * 1000
        fires_3857["geometry"] = fires_3857.buffer(buffer_m)

        # Spatial join: buildings intersecting any fire buffer
        # Keep useful FIRMS attributes for context
        fires_attrs = fires_3857[["geometry", "acq_date", "acq_time", "frp", "confidence"]]

        at_risk = gpd.sjoin(
            buildings,
            fires_attrs,
            how="inner",
            predicate="intersects",
        )

        if at_risk.empty:
            print(f"No buildings found within {BUFFER_KM} km of any NOAA-21 detection.")
        else:
            # Drop duplicates (same building hit by multiple overlapping buffers)
            at_risk = at_risk.reset_index().drop_duplicates(subset=["index"])

            print(f"Buildings within {BUFFER_KM} km of a FIRMS detection:", len(at_risk))

            # Keep a tidy set of columns for review
            cols_to_keep = [c for c in at_risk.columns if c not in ("geometry", "index_right")]
            summary = at_risk[cols_to_keep]

            # Show top rows
            display(summary.head(20))

            # Save for mapping / further analysis
            out_path = "/content/buildings_at_risk.geojson"
            at_risk.to_crs("EPSG:4326").to_file(out_path, driver="GeoJSON")
            print("Saved buildings at risk to:", out_path)


AOI bounds (lon/lat): -123.15210388183608,46.569071525412866,-122.79697329257401,46.805004817883024
Number of buildings: 23914
Requesting FIRMS data from: https://firms.modaps.eosdis.nasa.gov/api/area/csv/33d36beb1c667fdb1ed4e4b877c1cc23/VIIRS_NOAA21_NRT/-123.15210388183608,46.569071525412866,-122.79697329257401,46.805004817883024/2
No FIRMS detections from NOAA-21 in AOI bbox over last 2 day(s).
